# 03 — パック比エネルギー・質量・DC電力量

質量はBattery APIが元sourceの計算文を呼び、Energy APIは正式Controlled-DC関数をそのまま公開します。計算式・freezeは変更せず、資料書込みや本番実験を起動しません。

**実行方法**：[統合ガイドの実行手順](../docs/QUANTITATIVE_GUIDE.md#10-quickstart-and-dependencies)の環境を用意し、上からRun All。作業ディレクトリはrepo rootまたはnotebooks。第三者の住所・大容量入力・SUMOは不要です。途中の条件を編集したら、そのセル以降を再実行してください。正式実験条件・freezeは変更しません。

**前提・範囲**：Python 3.12系で確認する小型教材／保存結果表示です。必要な配送を省略した電力量減を省エネ成果としません。量子実機・量子優位性・全需要充足・全システムエネルギーの実証ではありません。Colab実行は未検証です。


In [1]:
from pathlib import Path
import sys, json, math
from decimal import Decimal
import numpy as np
import pandas as pd
from IPython.display import display
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'notebooks/adapters.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('リポジトリrootまたはnotebooksを作業ディレクトリにしてください')
sys.dont_write_bytecode = True
sys.path.insert(0, str(ROOT / 'notebooks'))
import adapters as api
S = '01_research_design/scenario_evaluation/'
M = '01_research_design/methodology/'
Q = '05_src/traffic_simulation/'
L = '05_src/literature_analysis/'


In [2]:
battery, scenarios=api.battery_api()
freeze=scenarios.load_model()
rows=api.official_mass_stages()
display(pd.DataFrame(rows)[['stage_id','e_pack_Wh_kg','E_pack_fixed_kWh','m_pack_model_kg','m_vehicle_model_kg']])
for row,stage in zip(rows,freeze['stage_inputs']):
    assert math.isclose(row['m_pack_model_kg'],stage['m_pack_kg'],abs_tol=1e-10)
    assert math.isclose(row['m_pack_model_kg'],float(Decimal(20000)/Decimal(str(row['e_pack_Wh_kg']))),abs_tol=1e-10)
    assert math.isclose(row['m_vehicle_model_kg'],stage['vehicle_mass_kg'],abs_tol=1e-10)

,stage_id,e_pack_Wh_kg,E_pack_fixed_kWh,m_pack_model_kg,m_vehicle_model_kg
0,B0,180,20,111.111111,1100.000000
1,B1,220,20,90.909091,1079.797980
2,B2,250,20,80.000000,1068.888889


## 小型走行入力
20kWh名目・18kWh初期利用可能。e_pack=180/220/250Wh/kg。パック軽量化の結果として車両質量が変わります。人工速度列は1秒刻み・定速10m/sで、SUMOの保存結果ではありません。
入力CSVは既存speed.jsonを同じ数値・順序で表現した人工fixtureです。Simulation APIが正式sourceのCSV読込みを呼び、返却した時刻・速度を変更せず既存Controlled-DCへ渡します。保存SUMO実測軌跡の代用や全量再現ではありません。


In [3]:
dc=api.energy_api()
simulation=api.simulation_api()
t,v=simulation.read_dc_input(api.example_input('trajectory_fixture'))
energy_rows=[]
for stage in freeze['stage_inputs']:
    for p in [4.5,2.6]:
        out,_=dc.evaluate(t,v,freeze,p,stage)
        dc.scalar_check(t,v,freeze,p,stage,out)
        expected=(Decimal(str(stage['running_mass_kg']))*Decimal('9.80665')*(Decimal('.005')+(Decimal('.01')+Decimal('.0095')*(Decimal(36)/100)**2)/Decimal(str(p)))*30+Decimal('.5')*Decimal('1.2041')*Decimal('1.175')*100*30)/3600/Decimal('.9')
        assert math.isclose(out['net_DC_Wh'],float(expected),rel_tol=1e-12)
        energy_rows.append({'stage':stage['stage_id'],'p_bar':p,'running_mass_kg':stage['running_mass_kg'],'DC_Wh':out['net_DC_Wh']})
display(pd.DataFrame(energy_rows))

,stage,p_bar,running_mass_kg,DC_Wh
0,B0,4.5,1155.000000,1.441145
1,B0,2.6,1155.000000,1.632426
2,B1,4.5,1134.797980,1.427395
3,B1,2.6,1134.797980,1.615330
4,B2,4.5,1123.888889,1.419970
5,B2,2.6,1123.888889,1.606099


## 入力を変更する
容量は正式20kWhを固定し、比エネルギー230Wh/kgと速度係数0.8を例示します。これは教育用の追加感度条件で、凍結研究結果ではありません。質量式は正式sourceの文を呼びます。

In [4]:
specific_energy=230.0  # Wh/kg、編集可能
speed_factor=0.8  # 0以上
assert np.isfinite(speed_factor) and speed_factor>=0
custom=api.mass_at_specific_energy(specific_energy)
stage={'running_mass_kg':custom['m_vehicle_model_kg']+freeze['running_mass_definition']['driver_common_kg']}
custom_result,_=dc.evaluate(t,v*speed_factor,freeze,4.5,stage)
display(pd.DataFrame([{'specific_energy_Wh_kg':specific_energy,'pack_kg':custom['m_pack_model_kg'],'vehicle_kg':custom['m_vehicle_model_kg'],'DC_Wh':custom_result['net_DC_Wh']}]))
# 非1秒刻みは拒否される
try: dc.evaluate(np.array([0,2,3,4]),v,freeze,4.5,stage)
except AssertionError: print('非1秒刻みを検出')
else: raise AssertionError('sampling violation not rejected')
assert math.isclose(custom['m_pack_model_kg'], float(Decimal(20000)/Decimal(str(specific_energy))), abs_tol=1e-10)
try: api.mass_at_specific_energy(0)
except ValueError: print('非正の比エネルギーを拒否')
else: raise AssertionError('invalid specific energy accepted')


,specific_energy_Wh_kg,pack_kg,vehicle_kg,DC_Wh
0,230.0,86.956522,1075.845411,0.94303


非1秒刻みを検出
非正の比エネルギーを拒否


## 読取り方と限界
固定した各軌跡で質量感度を評価します。平坦・回生なし・補機なし・エネルギー側積載0kg、運転者55kg、回転等価質量61kgは段階共通です。定速の独立期待値は転がり＋空気抵抗の30mの仕事/0.9です。観測外エネルギーは未知。量子材料計算・実用電池性能の達成・実車航続距離を実証していません。